In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

## 1. Data Loading 

In [ ]:

files = ["2011.csv","2012.csv","2013.csv","2014.csv","2015.csv"]
dfs = {year : pd.read_csv(f"C:/Users/Vansh/Desktop/Project/data/raw/{f}") for year,f in zip(range(2011,2016),files)}

for year,df in dfs.items():
    print(f"{year}:{df.shape} - {list(df.columns)}")


In [ ]:
common_cols = set(dfs[2011].columns)
for df in dfs.values():
    common_cols = common_cols.intersection(set(df.columns))

print(f"Common columns: {len(common_cols)}")
print(common_cols)

In [4]:
for year,df in dfs.items():
    df["Survey_year"] = year

## 2. Data Merging 

In [5]:
merged_df = pd.concat([df[list(common_cols)+["Survey_year"]] for df in dfs.values()],ignore_index=True)

merged_df.to_csv("C:/Users/Vansh/Desktop/Project/data/raw/merged.csv",index=False)

In [ ]:
print(merged_df.shape)
print(merged_df.head())

In [ ]:
if "DIABETE3" in list(merged_df.columns):
    print(True)

print(merged_df['DIABETE3'].value_counts())

merged_df["DIABETE3"] = merged_df["DIABETE3"].replace({7.0:np.nan,9.0:np.nan})
merged_df = merged_df.dropna(subset=["DIABETE3"])
print(merged_df.shape)
print(merged_df["DIABETE3"].value_counts())


## 3. Target Variable Analysis

In [ ]:
mapping = {1.0:1,2.0:1,4.0:1,3.0:0}
merged_df["Diabetes_binary"] = merged_df["DIABETE3"].map(mapping)
print(merged_df[["DIABETE3","Diabetes_binary"]])
print(merged_df["Diabetes_binary"].value_counts())

### Decision Documenting for Target Variable
1) The original values in the "DIABETE3" column were 1,2,3,4,7,9 where 
    - 1: Diabetic Patients
    - 2: Diabetic (While Preganant)
    - 3: Not Diabetic
    - 4: Pre Diabetic 
    - 7,9 were sentinel values showing "Refused" or "Not Answered"
2) We dropped 7 and 9 as they were a classic example of "Coded by the data Collector". Dropping these rows results in a loss of only 5,233 rows (~0.17% of data) — negligible at this scale. These values werent significant for our model training as they are irrelevant for classification
3) We Mapped 4 → 1 as this system is a risk prediction tool, not a diagnosis tool.Flagging pre-diabetics as high risk aligns with my stated goal of enabling lifestyle intervention before disease onset.
4) The final Class Distribution is
    - Class 0: 2010686 (84%)
    - Class 1: 364128 (16%)
5) This is a high class imbalance. I plan to handle this imbalance using mainly SMOTE for fixing the data , using "class_weight='balanced'" for fixing the model and lowering the decision threshold (flagging more people as positive) for fixing the threshold

In [9]:
merged_df.to_csv("C:/Users/Vansh/Desktop/Project/data/processed/merged_clean.csv",index=False)